In [1]:
import sys
import os
import multiprocessing

# CRITICAL: Set multiprocessing start method to 'spawn' BEFORE any CUDA initialization
# This fixes the "Cannot re-initialize CUDA in forked subprocess" error in Jupyter
multiprocessing.set_start_method('spawn', force=True)

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm import LLM, SamplingParams
from vllm_wrapper import SimpleLLMWrapper

# Import the original SAFE implementation
from third_party.factscore import atomic_facts
import itertools

def get_atomic_facts_safe(response: str, model, debug=False):
    """Wrapper that uses the original SAFE implementation with correct paths."""
    demon_dir = os.path.join(lff_root, "third_party", "factscore", "demos")
    atomic_fact_generator = atomic_facts.AtomicFactGenerator(
        api_key='', 
        demon_dir=demon_dir,
        gpt3_cache_file='', 
        other_lm=model
    )
    
    # Monkey patch the generate method to print prompts if debug=True
    if debug:
        original_generate = model.generate
        def debug_generate(prompt, **kwargs):
            print("="*80)
            print("PROMPT SENT TO MODEL:")
            print("="*80)
            print(prompt)
            print("="*80)
            result = original_generate(prompt, **kwargs)
            print("\nMODEL RESPONSE:")
            print("="*80)
            print(result)
            print("="*80)
            return result
        model.generate = debug_generate
    
    facts, _ = atomic_fact_generator.run(response)
    
    # Restore original generate if we patched it
    if debug:
        model.generate = original_generate
    
    # Convert to dict format
    facts_as_dict = [
        {'sentence': sentence, 'atomic_facts': identified_atomic_facts}
        for sentence, identified_atomic_facts in facts
    ]
    all_atomic_facts_list = list(
        itertools.chain.from_iterable([f['atomic_facts'] for f in facts_as_dict])
    )
    
    return {
        'num_claims': len(all_atomic_facts_list),
        'sentences_and_atomic_facts': facts,
        'all_atomic_facts': facts_as_dict,
    }


/root/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dummy_text_to_atomize = "Paris is the capital of France and Germany is a big but awful country. Are you green? SLUUUUUUURMMM hehehe? SCUUUUUUUM GAAAAAANG"
llm = SimpleLLMWrapper()
atomized = get_atomic_facts_safe(dummy_text_to_atomize, llm, debug=False)

print(atomized)

[2025-11-24 00:34:53] INFO _client.py:1025: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


 Paris is the capital of France and Germany is a big but awful country.

{"atomic_facts": [
  "- Paris is the capital of France.",
  "- Germany is a country.",
  "- Germany is big.",
  "- Germany is awful."
]}


[2025-11-24 00:34:55] INFO _client.py:1025: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


 Are you green?

{"atomic_facts": []}


[2025-11-24 00:34:56] INFO _client.py:1025: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


 SLUUUUUUURMMM hehehe?

{ "atomic_facts": [] }


[2025-11-24 00:35:01] INFO _client.py:1025: HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


 SCUUUUUUUM GAAAAAANG

{"atomic_facts": []}
{'num_claims': 4, 'sentences_and_atomic_facts': [('Paris is the capital of France and Germany is a big but awful country.', ['Paris is the capital of France.', 'Germany is a country.', 'Germany is big.', 'Germany is awful.']), ('Are you green?', []), ('SLUUUUUUURMMM hehehe?', []), ('SCUUUUUUUM GAAAAAANG', [])], 'all_atomic_facts': [{'sentence': 'Paris is the capital of France and Germany is a big but awful country.', 'atomic_facts': ['Paris is the capital of France.', 'Germany is a country.', 'Germany is big.', 'Germany is awful.']}, {'sentence': 'Are you green?', 'atomic_facts': []}, {'sentence': 'SLUUUUUUURMMM hehehe?', 'atomic_facts': []}, {'sentence': 'SCUUUUUUUM GAAAAAANG', 'atomic_facts': []}]}


In [4]:
import json
import os

responses = []
with open(os.getcwd() + "/data/responses.jsonl", "r") as f:
    for line in f:
        responses.append(json.loads(line))

# print one of the responses
print(responses[0]["responses"][0])



Sure! Here is a bio for Kang Ji-Hwan:

Kang Ji-Hwan is a South Korean singer-songwriter who gained popularity with his group After School. In recent years, he has pursued a solo career, releasing albums such as "Dynamite" (2017), "I Am" (2018), and "Love Is Magic" (2019). He has collaborated with various artists, including BTS' J-Hope, and has performed at numerous events both domestically and internationally. Additionally, Kang Ji-Hwan has ventured into hosting roles, appearing on shows like "Street Woman Fighter" and "Rookie Cops."


In [6]:
import threading

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()

def worker(response, model):
    """Worker function that collects results"""
    result = get_atomic_facts_safe(response, model)
    with results_lock:
        results.append(result)

threads = []
# Fix: enumerate returns (index, item), so iterate directly
for query in responses[:100]:
    for response in query["responses"]:
        t = threading.Thread(target=worker, args=(response, llm))
        t.start()
        threads.append(t)

for t in threads:
    t.join()

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.





In [7]:
total_number_of_facts = 0
total_number_of_sentences = 0
for result in results:
    total_number_of_facts += result['num_claims']
    total_number_of_sentences += len(result['sentences_and_atomic_facts'])

print(f"Total number of facts: {total_number_of_facts}")
print(f"Total number of sentences: {total_number_of_sentences}")
import random
# print a random sentence and its atomic facts
random_response = random.choice(results)
random_sentence = random.choice(random_response['sentences_and_atomic_facts'])
print(f"Random sentence: {random_sentence[0]}")
print(f"Atomic facts: {random_sentence[1]}")


Total number of facts: 18523
Total number of sentences: 5625
Random sentence: Assistant: Shashank Manohar is a prominent figure in Indian cricket administration and politics.
Atomic facts: ['Shashank Manohar is a prominent figure in Indian cricket administration.', 'Shashank Manohar is a prominent figure in Indian politics.']


In [10]:
with open("data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")